# CuPyCCx — GPU test on Google Colab

**Before running:** set runtime to GPU via *Runtime → Change runtime type → T4 GPU*

In [ ]:
# Cell 1 — confirm GPU is available
!nvidia-smi
!nvcc --version

In [ ]:
%%bash
# Cell 2 — install dependencies; remove any stale cupyccx install
apt-get install -qq cmake ninja-build libeigen3-dev libopenblas-dev

# The Colab base image ships a stale Debian intel-mkl 2020.4.304 (unversioned
# .so files in /lib/x86_64-linux-gnu) alongside a much newer pip `mkl`
# package pulled in by something else preinstalled on the image (versioned
# libmkl_*.so.2 files in /usr/local/lib). MKL's own runtime dispatcher
# (libmkl_rt.so) dlopens its kernel libraries by unversioned name, so the new
# dispatcher can end up pairing with the OLD kernel libs and hit a missing
# symbol, hard-crashing the whole kernel process (not a catchable Python
# exception):
#   Intel MKL FATAL ERROR: Cannot load libmkl_avx512.so or libmkl_def.so.
#   INTEL MKL ERROR: ... undefined symbol: mkl_sparse_optimize_bsr_trsm_i8.
# Nothing here needs system MKL — numpy/scipy ship their own bundled
# OpenBLAS and our C++ build resolves BLAS via find_package(BLAS) — so
# remove the stale apt intel-mkl entirely instead of trying to reconcile
# the two versions.
MKL_PKGS=$(dpkg-query -W -f='${Package}\n' 2>/dev/null | grep -E '^(intel-mkl|libmkl)') || true
if [ -n "$MKL_PKGS" ]; then
  apt-get purge -qq -y $MKL_PKGS
  apt-get autoremove -qq -y
  ldconfig
fi

# Force OpenBLAS to be the active libblas.so.3/liblapack.so.3 alternative
# too, as a second line of defense.
for name in libblas.so.3-x86_64-linux-gnu liblapack.so.3-x86_64-linux-gnu; do
  alt=$(update-alternatives --list "$name" 2>/dev/null | grep -i openblas | head -1)
  if [ -n "$alt" ]; then
    update-alternatives --set "$name" "$alt"
    echo "Set $name -> $alt"
  fi
done

pip install -q --upgrade pip
pip install -q pybind11 pyscf
pip uninstall -q -y cupyccx 2>/dev/null || true

In [ ]:
%%bash
# Cell 2b — DIAGNOSTIC: identify what's providing/loading the broken MKL libs.
# Run this once and share the output — it tells us whether the alternatives
# fix in Cell 2 actually applied, and which package owns the broken .so files.
echo "--- broken MKL files ---"
ls -la /lib/x86_64-linux-gnu/libmkl_avx512.so /lib/x86_64-linux-gnu/libmkl_def.so \
       /lib/x86_64-linux-gnu/libmkl_core.so /lib/x86_64-linux-gnu/libmkl_rt.so* 2>&1

echo; echo "--- owning packages ---"
dpkg -S /lib/x86_64-linux-gnu/libmkl_avx512.so /lib/x86_64-linux-gnu/libmkl_def.so \
        /lib/x86_64-linux-gnu/libmkl_core.so 2>&1

echo; echo "--- all installed mkl* packages ---"
dpkg -l | grep -i mkl

echo; echo "--- pip mkl* packages ---"
pip list 2>/dev/null | grep -i mkl

echo; echo "--- BLAS/LAPACK alternatives after Cell 2 ---"
update-alternatives --display libblas.so.3-x86_64-linux-gnu 2>&1
update-alternatives --display liblapack.so.3-x86_64-linux-gnu 2>&1

echo; echo "--- ldconfig cache mkl entries ---"
ldconfig -p | grep -i mkl

echo; echo "--- numpy/scipy BLAS config ---"
python3 -c "import numpy; numpy.show_config()" 2>&1
python3 -c "import scipy; scipy.show_config()" 2>&1 | head -40

In [ ]:
%%bash
# Cell 3 — clone and build with CUDA (T4 = sm_75; A100 = sm_80)
# Always start from /content so re-running doesn't nest CuPyCCx/CuPyCCx/...
cd /content

rm -rf CuPyCCx
git clone --quiet https://github.com/varunrishi/CuPyCCx.git
cd CuPyCCx

# Use the system cmake from apt (/usr/bin/cmake), NOT the pip-installed cmake
# which cannot locate nvcc. pip install pyscf pulls in cmake 3.31 as a
# Python package and it shadows the system cmake on PATH.
/usr/bin/cmake -B build \
  -DCUPYCCX_CUDA=ON \
  -DCUPYCCX_CUDA_ARCH=75 \
  -DCMAKE_BUILD_TYPE=Release \
  -DCUPYCCX_BUILD_TESTS=OFF \
  -Dpybind11_DIR=$(python3 -c "import pybind11; print(pybind11.get_cmake_dir())")
/usr/bin/cmake --build build -j$(nproc)

# Install Python files + CUDA extension directly into site-packages
SITE=$(python3 -c "import site; print(site.getsitepackages()[0])")
rm -rf "$SITE/cupyccx"
cp -r python/cupyccx "$SITE/"
cp build/_cupyccx*.so "$SITE/cupyccx/"
echo "Installed to: $SITE/cupyccx/"
ls "$SITE/cupyccx/"

# Verify import works before leaving bash
python3 -c "import importlib; importlib.invalidate_caches(); import cupyccx._cupyccx; print('Extension OK:', cupyccx._cupyccx.__file__)"

In [ ]:
# Cell 4 — verify extension loads in the notebook kernel
import sys, importlib, os, glob

# Evict all stale cupyccx modules
for key in list(sys.modules):
    if 'cupyccx' in key:
        del sys.modules[key]

# Find the installed .so and promote its site-packages to the front of sys.path.
# This handles the case where a stale editable install elsewhere on sys.path
# shadows the freshly built extension.
matches = glob.glob('/usr/local/lib/python*/dist-packages/cupyccx/_cupyccx*.so')
if matches:
    site_dir = os.path.dirname(os.path.dirname(matches[0]))
    sys.path = [site_dir] + [p for p in sys.path if p != site_dir]
    print(f'Using site-packages: {site_dir}')
else:
    print('WARNING: could not find _cupyccx*.so under /usr/local/lib — Cell 3 may not have completed')

importlib.invalidate_caches()

import cupyccx._cupyccx
print('Extension loaded from:', cupyccx._cupyccx.__file__)

In [ ]:
import time
from pyscf import gto, scf
from cupyccx.scf_data import prepare_from_pyscf
from cupyccx.method import CCD, CCOptions

# Cell 5 — N2/STO-3G: CPU vs GPU correctness check
mol  = gto.M(atom='N 0 0 0; N 0 0 2.118', basis='sto-3g', unit='Bohr', verbose=0)
mf   = scf.RHF(mol).run()
data = prepare_from_pyscf(mf, verbose=False)

t0 = time.time()
r_cpu = CCD.from_scf_data(data, opts=CCOptions(use_gpu=False, max_iter=200, conv_energy=1e-9, conv_amp=1e-8)).compute(e_scf=data.e_scf)
t_cpu = time.time() - t0

t0 = time.time()
r_gpu = CCD.from_scf_data(data, opts=CCOptions(use_gpu=True,  max_iter=200, conv_energy=1e-9, conv_amp=1e-8)).compute(e_scf=data.e_scf)
t_gpu = time.time() - t0

print(f'CPU  E_corr = {r_cpu.e_corr:.12f} Ha  ({t_cpu:.2f}s)')
print(f'GPU  E_corr = {r_gpu.e_corr:.12f} Ha  ({t_gpu:.2f}s)')
print(f'Diff        = {abs(r_cpu.e_corr - r_gpu.e_corr):.2e} Ha')

In [ ]:
# Cell 6 — larger system (cc-pVDZ) to see GPU speedup
mol2  = gto.M(atom='N 0 0 0; N 0 0 2.118', basis='cc-pVDZ', unit='Bohr', verbose=0)
mf2   = scf.RHF(mol2).run()
data2 = prepare_from_pyscf(mf2, verbose=False)
print(f'n_occ={data2.n_occ}  n_vir={data2.n_vir}  n_mo={data2.n_mo}')

t0 = time.time()
r2_cpu = CCD.from_scf_data(data2, opts=CCOptions(use_gpu=False, max_iter=200, conv_energy=1e-9, conv_amp=1e-8)).compute(e_scf=data2.e_scf)
t_cpu = time.time() - t0

t0 = time.time()
r2_gpu = CCD.from_scf_data(data2, opts=CCOptions(use_gpu=True,  max_iter=200, conv_energy=1e-9, conv_amp=1e-8)).compute(e_scf=data2.e_scf)
t_gpu = time.time() - t0

print(f'CPU  E_corr = {r2_cpu.e_corr:.12f} Ha  ({t_cpu:.2f}s)')
print(f'GPU  E_corr = {r2_gpu.e_corr:.12f} Ha  ({t_gpu:.2f}s)')
print(f'Speedup     = {t_cpu/t_gpu:.1f}x')

In [ ]:
import time
from cupyccx.method import DCD, pCCD, CCOptions

# Cell 7 — DCD and pCCD CPU vs GPU correctness check (N2/STO-3G)
# data is already prepared in Cell 5
opts_cpu = CCOptions(use_gpu=False, max_iter=200, conv_energy=1e-9, conv_amp=1e-8)
opts_gpu = CCOptions(use_gpu=True,  max_iter=200, conv_energy=1e-9, conv_amp=1e-8)

print('--- DCD ---')
r_dcd_cpu = DCD.from_scf_data(data, opts=opts_cpu).compute(e_scf=data.e_scf)
r_dcd_gpu = DCD.from_scf_data(data, opts=opts_gpu).compute(e_scf=data.e_scf)
diff_dcd  = abs(r_dcd_cpu.e_corr - r_dcd_gpu.e_corr)
print(f'CPU  E_corr = {r_dcd_cpu.e_corr:.12f} Ha')
print(f'GPU  E_corr = {r_dcd_gpu.e_corr:.12f} Ha')
print(f'Diff        = {diff_dcd:.2e} Ha  {"OK" if diff_dcd < 1e-7 else "FAIL"}')

print()
print('--- pCCD(alpha=1, beta=1) — must equal CCD ---')
r_pccd_cpu = pCCD.from_scf_data(data, opts=opts_cpu, alpha=1.0, beta=1.0).compute(e_scf=data.e_scf)
r_pccd_gpu = pCCD.from_scf_data(data, opts=opts_gpu, alpha=1.0, beta=1.0).compute(e_scf=data.e_scf)
diff_pccd  = abs(r_pccd_cpu.e_corr - r_pccd_gpu.e_corr)
print(f'CPU  E_corr = {r_pccd_cpu.e_corr:.12f} Ha')
print(f'GPU  E_corr = {r_pccd_gpu.e_corr:.12f} Ha')
print(f'Diff        = {diff_pccd:.2e} Ha  {"OK" if diff_pccd < 1e-7 else "FAIL"}')
print(f'pCCD vs CCD = {abs(r_pccd_cpu.e_corr - r_cpu.e_corr):.2e} Ha  (should be ~0)')


In [ ]:
%%bash
# Cell 8 — install CuPy (matches Colab's CUDA 12.x) and pytest
# cupy-cuda12x supports CUDA 12.0–12.x; swap for cupy-cuda11x on CUDA 11 runtimes
pip install -q cupy-cuda12x pytest
python3 -c "import cupy; print('CuPy', cupy.__version__, '— CUDA', cupy.cuda.runtime.runtimeGetVersion())"

In [ ]:
%%bash
# Cell 9 — run all Python tests
# Covers test_ccd.py, test_pyscf.py (C++ extension), and test_py_solver.py
# (PyCCD/PyLCCD/PyDCD/PypCCD including CuPy GPU path)
cd /content/CuPyCCx
python3 -m pytest tests/python/ -v 2>&1

In [ ]:
import time, cupy as cp
from cupyccx.method import CCD, DCD, CCOptions
from cupyccx.py_solver import PyCCD, PyDCD

# Cell 10 — PyCCD (CuPy GPU) vs C++ CCD: energy + T2 amplitude round-trip
# data is already prepared in Cell 5
print('=== PyCCD (CuPy) vs C++ CCD — N2/STO-3G ===')
opts_gpu_py = CCOptions(use_gpu=True, max_iter=200, conv_energy=1e-9, conv_amp=1e-8)
opts_gpu_cpp = CCOptions(use_gpu=True, max_iter=200, conv_energy=1e-9, conv_amp=1e-8)

t0 = time.time()
r_py = PyCCD.from_scf_data(data, opts=opts_gpu_py).compute(data.e_scf)
t_py = time.time() - t0

t0 = time.time()
r_cpp = CCD.from_scf_data(data, opts=opts_gpu_cpp).compute(data.e_scf)
t_cpp = time.time() - t0

import numpy as np
e_diff  = abs(r_py.e_corr - r_cpp.e_corr)
t2_diff = np.max(np.abs(r_py.t2 - r_cpp.t2))

print(f'PyCCD(CuPy)  E_corr = {r_py.e_corr:.12f} Ha  ({t_py:.2f}s)')
print(f'C++ CCD      E_corr = {r_cpp.e_corr:.12f} Ha  ({t_cpp:.2f}s)')
print(f'Energy diff  = {e_diff:.2e} Ha  {"OK" if e_diff < 1e-8 else "FAIL"}')
print(f'T2 max diff  = {t2_diff:.2e}    {"OK" if t2_diff < 1e-8 else "FAIL"}')

print()
print('=== PyDCD (CuPy) vs C++ DCD — N2/STO-3G ===')
t0 = time.time()
r_dcd_py = PyDCD.from_scf_data(data, opts=opts_gpu_py).compute(data.e_scf)
t_py2 = time.time() - t0

t0 = time.time()
r_dcd_cpp = DCD.from_scf_data(data, opts=opts_gpu_cpp).compute(data.e_scf)
t_cpp2 = time.time() - t0

e_diff2 = abs(r_dcd_py.e_corr - r_dcd_cpp.e_corr)
print(f'PyDCD(CuPy)  E_corr = {r_dcd_py.e_corr:.12f} Ha  ({t_py2:.2f}s)')
print(f'C++ DCD      E_corr = {r_dcd_cpp.e_corr:.12f} Ha  ({t_cpp2:.2f}s)')
print(f'Energy diff  = {e_diff2:.2e} Ha  {"OK" if e_diff2 < 1e-8 else "FAIL"}')

In [ ]:
%%bash
# Cell 8 — Nsight Systems profile of GPU tensor contractions
# Generates cupyccx_profile.nsys-rep; download and open in Nsight Systems UI.
#
# Nsight Compute (per-kernel metrics) — uncomment to use instead:
#   "$NCU" --set full --target-processes all -o /content/cupyccx_ncu \
#       python3 /content/CuPyCCx/examples/profile_gpu.py

# nsys is not on PATH in Colab — find it under the CUDA toolkit
NSYS=$(find /usr/local/cuda /opt/nvidia -name nsys 2>/dev/null | head -1)
NCU=$(find /usr/local/cuda /opt/nvidia -name ncu  2>/dev/null | head -1)
echo "nsys: $NSYS"
echo "ncu:  $NCU"

"$NSYS" profile \
  --trace=cuda,cublas,osrt \
  --output=/content/cupyccx_profile \
  --force-overwrite=true \
  python3 /content/CuPyCCx/examples/profile_gpu.py

echo
echo "Report written to /content/cupyccx_profile.nsys-rep"
echo "Download it and open in Nsight Systems (https://developer.nvidia.com/nsight-systems)"
